In [24]:
"""
==========================
Loads trained models + test data from Feature_Selected_Retrain.py artifacts.

Key change from v1: uses quantile binning (equal-count bins) instead of
uniform binning, and adds class-conditional probability histograms. The
uniform strategy was misleading bc test prevalence is ~1%, so 9/10 bins
had near-zero positives and ECE looked artificially good.

Produces:
  - per-model calibration curves (quantile bins, all folds)
  - combined overlay (last fold)
  - class-conditional predicted probability histograms (the real money plot)
  - ECE / Brier summary table
  - discrimination slope plot
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import os
import warnings
from scipy.sparse import load_npz
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss, roc_auc_score

warnings.filterwarnings('ignore')

In [25]:
# =============================================================
# CONFIG
# =============================================================
ARTIFACT_DIR = "shap_artifacts_feature_selected"
K_FOLDS = 5
N_BINS = 10
STRATEGY = 'quantile'       # equal-count bins — critical for low-prevalence data
OUTPUT_DIR = "calibration_results_V2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [26]:
# =============================================================
# HELPERS
# =============================================================
def expected_calibration_error(y_true, y_prob, n_bins=10, strategy='quantile'):
    """
    Weighted mean |accuracy - confidence| across bins.
    Uses calibration_curve output directly — no separate bin reconstruction,
    which avoids shape mismatches when quantile bins get merged.
    Weights each bin equally (unweighted ECE) since calibration_curve
    doesn't expose per-bin sample counts reliably across strategies.
    """
    for nb in [n_bins, 5, 3]:
        try:
            fraction_pos, mean_pred = calibration_curve(
                y_true, y_prob, n_bins=nb, strategy=strategy
            )
            break
        except ValueError:
            continue
    else:
        return 0.0

    # unweighted average — each returned bin gets equal weight.
    # slightly less precise than sample-weighted ECE but robust to
    # the bin-merging issue with quantile strategy.
    return np.mean(np.abs(fraction_pos - mean_pred))


def load_model(model_name, fold_idx):
    path = os.path.join(ARTIFACT_DIR, f"{model_name}_fold{fold_idx}.pkl")
    with open(path, "rb") as f:
        return pickle.load(f)


def get_probs(model_name, model, X_test):
    if model_name == 'lasso':
        lasso_model, scaler = model
        X_scaled = scaler.transform(X_test)
        return lasso_model.predict_proba(X_scaled)[:, 1]
    else:
        return model.predict_proba(X_test)[:, 1]


MODEL_SPECS = [
    ('dt',       'Decision Tree', '#2196F3'),
    ('lasso',    'Lasso',         '#FF5722'),
    ('xgb',      'XGBoost',       '#9C27B0'),
    ('rf_tuned', 'Tuned RF',      '#4CAF50'),
]

In [27]:
# =============================================================
# LOAD DATA
# =============================================================
print("=" * 60)
print("LOADING ARTIFACTS")
print("=" * 60)

feature_cols = np.load(os.path.join(ARTIFACT_DIR, "feature_cols.npy"), allow_pickle=True)
print(f"Features: {len(feature_cols)}")

test_data = []
for i in range(K_FOLDS):
    X = load_npz(os.path.join(ARTIFACT_DIR, f"X_test_fold{i}.npz"))
    y = np.load(os.path.join(ARTIFACT_DIR, f"y_test_fold{i}.npy"))
    test_data.append((X, y))
    prevalence = y.sum() / len(y) * 100
    print(f"  Fold {i}: {X.shape[0]} samples (pos={int(y.sum())}, "
          f"neg={int(len(y)-y.sum())}, prevalence={prevalence:.2f}%)")

LOADING ARTIFACTS
Features: 149
  Fold 0: 42189 samples (pos=16, neg=42173, prevalence=0.04%)
  Fold 1: 42188 samples (pos=15, neg=42173, prevalence=0.04%)
  Fold 2: 42188 samples (pos=15, neg=42173, prevalence=0.04%)
  Fold 3: 42188 samples (pos=15, neg=42173, prevalence=0.04%)
  Fold 4: 42188 samples (pos=15, neg=42173, prevalence=0.04%)


In [28]:
# =============================================================
# COMPUTE STATS — ALL MODELS, ALL FOLDS
# =============================================================
print("\n" + "=" * 60)
print("CALIBRATION ANALYSIS (quantile bins)")
print("=" * 60)

results = []

for model_key, model_name, color in MODEL_SPECS:
    print(f"\n--- {model_name} ---")

    fold_eces = []
    fold_briers = []
    fold_aucs = []

    for fold_idx in range(K_FOLDS):
        model = load_model(model_key, fold_idx)
        X_test, y_test = test_data[fold_idx]
        y_prob = get_probs(model_key, model, X_test)

        ece = expected_calibration_error(y_test, y_prob, N_BINS, STRATEGY)
        brier = brier_score_loss(y_test, y_prob)
        auc = roc_auc_score(y_test, y_prob)

        fold_eces.append(ece)
        fold_briers.append(brier)
        fold_aucs.append(auc)

        print(f"  Fold {fold_idx}: ECE={ece:.4f}  Brier={brier:.4f}  AUC={auc:.4f}")

    results.append({
        'model': model_name,
        'ece_mean': np.mean(fold_eces),
        'ece_std': np.std(fold_eces),
        'brier_mean': np.mean(fold_briers),
        'brier_std': np.std(fold_briers),
        'auc_mean': np.mean(fold_aucs),
        'auc_std': np.std(fold_aucs),
    })

    print(f"  Mean:  ECE={np.mean(fold_eces):.4f}±{np.std(fold_eces):.4f}  "
          f"Brier={np.mean(fold_briers):.4f}±{np.std(fold_briers):.4f}  "
          f"AUC={np.mean(fold_aucs):.4f}±{np.std(fold_aucs):.4f}")

results_df = pd.DataFrame(results)
results_df.to_csv(os.path.join(OUTPUT_DIR, "calibration_summary_v2.csv"), index=False)
print(f"\nSaved: calibration_summary_v2.csv")


CALIBRATION ANALYSIS (quantile bins)

--- Decision Tree ---
  Fold 0: ECE=0.4249  Brier=0.0548  AUC=0.7815
  Fold 1: ECE=0.5628  Brier=0.1119  AUC=0.6890
  Fold 2: ECE=0.5735  Brier=0.1181  AUC=0.6116
  Fold 3: ECE=0.5517  Brier=0.0909  AUC=0.5953
  Fold 4: ECE=0.5774  Brier=0.1055  AUC=0.7739
  Mean:  ECE=0.5381±0.0573  Brier=0.0962±0.0226  AUC=0.6903±0.0781

--- Lasso ---
  Fold 0: ECE=0.0543  Brier=0.0376  AUC=0.8978
  Fold 1: ECE=0.0765  Brier=0.0516  AUC=0.7254
  Fold 2: ECE=0.0793  Brier=0.0517  AUC=0.6924
  Fold 3: ECE=0.0740  Brier=0.0491  AUC=0.8927
  Fold 4: ECE=0.0746  Brier=0.0491  AUC=0.7526
  Mean:  ECE=0.0718±0.0089  Brier=0.0478±0.0052  AUC=0.7922±0.0863

--- XGBoost ---
  Fold 0: ECE=0.0036  Brier=0.0012  AUC=0.8701
  Fold 1: ECE=0.0047  Brier=0.0019  AUC=0.8119
  Fold 2: ECE=0.0036  Brier=0.0013  AUC=0.8586
  Fold 3: ECE=0.0037  Brier=0.0013  AUC=0.7347
  Fold 4: ECE=0.0038  Brier=0.0015  AUC=0.8103
  Mean:  ECE=0.0039±0.0004  Brier=0.0014±0.0002  AUC=0.8171±0.0477



In [29]:
# =============================================================
# PLOT 1: CLASS-CONDITIONAL PROBABILITY HISTOGRAMS
# =============================================================
# This is the most informative plot for an oversampled screening model.
# Shows how well the model SEPARATES the two classes, regardless of
# whether the raw probabilities are calibrated to true prevalence.
print("\n" + "=" * 60)
print("PLOTTING: CLASS-CONDITIONAL PROBABILITY HISTOGRAMS")
print("=" * 60)

LAST_FOLD = K_FOLDS - 1
X_last, y_last = test_data[LAST_FOLD]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, (model_key, model_name, color) in enumerate(MODEL_SPECS):
    ax = axes[idx]
    model = load_model(model_key, LAST_FOLD)
    y_prob = get_probs(model_key, model, X_last)

    prob_neg = y_prob[y_last == 0]
    prob_pos = y_prob[y_last == 1]

    # use same bins for both classes
    bins = np.linspace(0, 1, 51)

    ax.hist(prob_neg, bins=bins, alpha=0.6, color='#1976D2', label='Control',
            density=True, edgecolor='white', linewidth=0.3)
    ax.hist(prob_pos, bins=bins, alpha=0.6, color='#D32F2F', label='Disease',
            density=True, edgecolor='white', linewidth=0.3)

    # discrimination slope = mean(P|pos) - mean(P|neg)
    disc_slope = prob_pos.mean() - prob_neg.mean()
    auc = roc_auc_score(y_last, y_prob)

    ax.set_xlabel('Predicted P(Disease)')
    ax.set_ylabel('Density')
    ax.set_title(f'{model_name}\n'
                 f'AUC={auc:.3f}  |  Disc. slope={disc_slope:.3f}  |  '
                 f'μ(neg)={prob_neg.mean():.3f}  μ(pos)={prob_pos.mean():.3f}',
                 fontsize=11, fontweight='bold', color=color)
    ax.legend(fontsize=10)
    ax.set_xlim([-0.02, 1.02])

plt.suptitle('Predicted Probability by True Class (Last Fold)',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "class_conditional_histograms.png"),
            dpi=150, bbox_inches='tight')
plt.close()
print("Saved: class_conditional_histograms.png")


PLOTTING: CLASS-CONDITIONAL PROBABILITY HISTOGRAMS
Saved: class_conditional_histograms.png


In [30]:
# =============================================================
# PLOT 2: CLASS-CONDITIONAL HISTOGRAMS — ALL FOLDS OVERLAID
# =============================================================
print("\n" + "=" * 60)
print("PLOTTING: CLASS-CONDITIONAL HISTOGRAMS (all folds)")
print("=" * 60)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, (model_key, model_name, color) in enumerate(MODEL_SPECS):
    ax = axes[idx]
    disc_slopes = []

    for fold_idx in range(K_FOLDS):
        model = load_model(model_key, fold_idx)
        X_test, y_test = test_data[fold_idx]
        y_prob = get_probs(model_key, model, X_test)

        prob_pos = y_prob[y_test == 1]
        prob_neg = y_prob[y_test == 0]
        disc_slopes.append(prob_pos.mean() - prob_neg.mean())

        # only plot disease class across folds (neg class looks the same)
        ax.hist(prob_pos, bins=np.linspace(0, 1, 51), alpha=0.3,
                density=True, label=f'Disease fold {fold_idx}' if fold_idx < 2 else None)

    # add fold 0 neg class for reference
    model0 = load_model(model_key, 0)
    y_prob0 = get_probs(model_key, model0, test_data[0][0])
    prob_neg0 = y_prob0[test_data[0][1] == 0]
    ax.hist(prob_neg0, bins=np.linspace(0, 1, 51), alpha=0.3, color='grey',
            density=True, label='Control (fold 0)')

    ax.set_xlabel('Predicted P(Disease)')
    ax.set_ylabel('Density')
    ax.set_title(f'{model_name}  |  Disc. slope={np.mean(disc_slopes):.3f}'
                 f'±{np.std(disc_slopes):.3f}',
                 fontsize=11, fontweight='bold', color=color)
    ax.legend(fontsize=8)
    ax.set_xlim([-0.02, 1.02])

plt.suptitle('Predicted Probability Stability Across Folds',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "class_conditional_allfolds.png"),
            dpi=150, bbox_inches='tight')
plt.close()
print("Saved: class_conditional_allfolds.png")


PLOTTING: CLASS-CONDITIONAL HISTOGRAMS (all folds)
Saved: class_conditional_allfolds.png


In [31]:
# =============================================================
# PLOT 3: CALIBRATION CURVES — QUANTILE BINS (last fold)
# =============================================================
print("\n" + "=" * 60)
print("PLOTTING: CALIBRATION CURVES (quantile bins, last fold)")
print("=" * 60)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Perfectly calibrated')

for model_key, model_name, color in MODEL_SPECS:
    model = load_model(model_key, LAST_FOLD)
    y_prob = get_probs(model_key, model, X_last)

    try:
        fraction_pos, mean_pred = calibration_curve(
            y_last, y_prob, n_bins=N_BINS, strategy='quantile'
        )
    except ValueError:
        fraction_pos, mean_pred = calibration_curve(
            y_last, y_prob, n_bins=5, strategy='quantile'
        )

    ece = expected_calibration_error(y_last, y_prob, N_BINS, STRATEGY)
    ax.plot(mean_pred, fraction_pos, 's-', color=color, lw=2, markersize=6,
            label=f'{model_name} (ECE={ece:.3f})')

ax.set_xlabel('Mean predicted probability (quantile bins)', fontsize=12)
ax.set_ylabel('Fraction of positives', fontsize=12)
ax.set_title('Calibration Curves — Quantile Binning (Last Fold)',
             fontsize=13, fontweight='bold')
ax.legend(loc='upper left', fontsize=9)

# zoom inset for the low-probability region where all the action is
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
ax_inset = inset_axes(ax, width="45%", height="45%", loc='center right')
ax_inset.plot([0, 0.15], [0, 0.15], 'k--', lw=1)

for model_key, model_name, color in MODEL_SPECS:
    model = load_model(model_key, LAST_FOLD)
    y_prob = get_probs(model_key, model, X_last)
    try:
        fraction_pos, mean_pred = calibration_curve(
            y_last, y_prob, n_bins=N_BINS, strategy='quantile'
        )
    except ValueError:
        fraction_pos, mean_pred = calibration_curve(
            y_last, y_prob, n_bins=5, strategy='quantile'
        )
    mask = mean_pred <= 0.15
    if mask.any():
        ax_inset.plot(mean_pred[mask], fraction_pos[mask], 's-', color=color,
                      lw=1.5, markersize=4)

ax_inset.set_xlim([0, 0.15])
ax_inset.set_ylim([0, 0.15])
ax_inset.set_title('Zoomed (0–0.15)', fontsize=9)
ax_inset.tick_params(labelsize=7)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "calibration_quantile_lastfold.png"),
            dpi=150, bbox_inches='tight')
plt.close()
print("Saved: calibration_quantile_lastfold.png")


PLOTTING: CALIBRATION CURVES (quantile bins, last fold)
Saved: calibration_quantile_lastfold.png


In [32]:
# =============================================================
# PLOT 4: PER-MODEL CALIBRATION ACROSS ALL FOLDS (quantile)
# =============================================================
print("\n" + "=" * 60)
print("PLOTTING: PER-MODEL CALIBRATION (all folds, quantile)")
print("=" * 60)

fig, axes = plt.subplots(2, 2, figsize=(14, 14))
axes = axes.flatten()

for idx, (model_key, model_name, color) in enumerate(MODEL_SPECS):
    ax = axes[idx]
    ax.plot([0, 1], [0, 1], 'k--', lw=1.5, alpha=0.5)

    for fold_idx in range(K_FOLDS):
        model = load_model(model_key, fold_idx)
        X_test, y_test = test_data[fold_idx]
        y_prob = get_probs(model_key, model, X_test)

        try:
            fraction_pos, mean_pred = calibration_curve(
                y_test, y_prob, n_bins=N_BINS, strategy='quantile'
            )
        except ValueError:
            fraction_pos, mean_pred = calibration_curve(
                y_test, y_prob, n_bins=5, strategy='quantile'
            )
        ece = expected_calibration_error(y_test, y_prob, N_BINS, STRATEGY)
        ax.plot(mean_pred, fraction_pos, 'o-', alpha=0.6,
                label=f'Fold {fold_idx} (ECE={ece:.3f})')

    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Fraction of positives')
    ax.set_title(f'{model_name}', fontsize=13, fontweight='bold', color=color)
    ax.legend(fontsize=8, loc='upper left')
    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.02])
    ax.set_aspect('equal')

plt.suptitle('Calibration Curves — All Models × All Folds (Quantile Bins)',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "calibration_per_model_allfolds_v2.png"),
            dpi=150, bbox_inches='tight')
plt.close()
print("Saved: calibration_per_model_allfolds_v2.png")


PLOTTING: PER-MODEL CALIBRATION (all folds, quantile)
Saved: calibration_per_model_allfolds_v2.png


In [33]:
# =============================================================
# PLOT 5: DISCRIMINATION SLOPE BAR CHART
# =============================================================
print("\n" + "=" * 60)
print("PLOTTING: DISCRIMINATION SLOPE COMPARISON")
print("=" * 60)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2196F3', '#FF5722', '#9C27B0', '#4CAF50']
x = np.arange(len(MODEL_SPECS))

disc_means = []
disc_stds = []

for model_key, model_name, color in MODEL_SPECS:
    slopes = []
    for fold_idx in range(K_FOLDS):
        model = load_model(model_key, fold_idx)
        X_test, y_test = test_data[fold_idx]
        y_prob = get_probs(model_key, model, X_test)
        slopes.append(y_prob[y_test == 1].mean() - y_prob[y_test == 0].mean())
    disc_means.append(np.mean(slopes))
    disc_stds.append(np.std(slopes))

bars = ax.bar(x, disc_means, yerr=disc_stds, color=colors, alpha=0.85,
              capsize=5, edgecolor='black', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels([s[1] for s in MODEL_SPECS], fontsize=11)
ax.set_ylabel('Discrimination Slope (higher = better separation)')
ax.set_title('Discrimination Slope: mean P(disease|pos) − mean P(disease|neg)',
             fontsize=13, fontweight='bold')
for bar, val in zip(bars, disc_means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "discrimination_slope_comparison.png"),
            dpi=150, bbox_inches='tight')
plt.close()
print("Saved: discrimination_slope_comparison.png")


PLOTTING: DISCRIMINATION SLOPE COMPARISON
Saved: discrimination_slope_comparison.png


In [34]:
# =============================================================
# PLOT 6: ECE + BRIER BAR CHART (updated w/ quantile ECE)
# =============================================================
print("\n" + "=" * 60)
print("PLOTTING: ECE + BRIER COMPARISON")
print("=" * 60)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

model_names = [r['model'] for r in results]
x = np.arange(len(model_names))

ax = axes[0]
ece_means = [r['ece_mean'] for r in results]
ece_stds = [r['ece_std'] for r in results]
bars = ax.bar(x, ece_means, yerr=ece_stds, color=colors, alpha=0.85,
              capsize=5, edgecolor='black', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=10)
ax.set_ylabel('ECE (lower = better)')
ax.set_title('Expected Calibration Error (quantile bins)', fontsize=13, fontweight='bold')
for bar, val in zip(bars, ece_means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)

ax = axes[1]
brier_means = [r['brier_mean'] for r in results]
brier_stds = [r['brier_std'] for r in results]
bars = ax.bar(x, brier_means, yerr=brier_stds, color=colors, alpha=0.85,
              capsize=5, edgecolor='black', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=10)
ax.set_ylabel('Brier Score (lower = better)')
ax.set_title('Brier Score', fontsize=13, fontweight='bold')
for bar, val in zip(bars, brier_means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "calibration_ece_brier_comparison_v2.png"),
            dpi=150, bbox_inches='tight')
plt.close()
print("Saved: calibration_ece_brier_comparison_v2.png")


PLOTTING: ECE + BRIER COMPARISON
Saved: calibration_ece_brier_comparison_v2.png


In [35]:
# =============================================================
# DONE
# =============================================================
print("\n" + "=" * 60)
print("ALL OUTPUTS")
print("=" * 60)
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fsize = os.path.getsize(os.path.join(OUTPUT_DIR, fname))
    print(f"  {fname:50s} ({fsize/1024:.1f} KB)")

print("\n" + "=" * 60)
print("DONE")
print("=" * 60)


ALL OUTPUTS
  calibration_ece_brier_comparison_v2.png            (56.8 KB)
  calibration_per_model_allfolds_v2.png              (173.0 KB)
  calibration_quantile_lastfold.png                  (90.2 KB)
  calibration_summary_v2.csv                         (0.6 KB)
  class_conditional_allfolds.png                     (112.2 KB)
  class_conditional_histograms.png                   (129.4 KB)
  discrimination_slope_comparison.png                (46.7 KB)

DONE
